## Analyse descriptive
# Introduction

Ce rapport présente une analyse descriptive complète des données horaires de qualité de l'air (concentration en PM2.5) collectées en Ile-de-France sur la période 2021-2025. L'objectif est d'identifier les tendances, la saisonnalité et les variations spatiales de la pollution aux particules fines.

Les données proviennent de 18 stations de mesure réparties en Ile-de-France, avec un focus particulier sur les zones urbaines, industrielles et rurales. Les variables analysées incluent la concentration en PM2.5, les conditions météorologiques (température, humidité, vitesse du vent, précipitations) et la densité d'installations industrielles.


In [ ]:
import sys
import os
import pandas as pd
sys.path.insert(0, os.path.abspath(".."))

## Importation de la base de données

In [ ]:
df = pd.read_csv("https://minio.lab.sspcloud.fr/ganlea/projet-qualite-air/processed/dataset_consolide.csv")
df["datetime_debut"]=pd.to_datetime(df["datetime_debut"])

## 1. ANALYSE DESCRITPIVE DE LA VARIABLE CIBLE (PM2.5)

On commence par une visualisation des types des variables contenues dans la base de données

In [ ]:
print(df.dtypes)

Une visualisation d'ensemble de chaque variable quantitative permet d'avoir une idée sur la distribution et le comportement de ces variables.

In [ ]:
liste_variables_quanti = ["pm25_brute", "vent_vitesse_ms", "vent_direction_deg", "temperature_c", "humidite_pct",
                            "pluie_mm", "nb_installations_5km"]
variables_quantitatives = df[liste_variables_quanti]
variables_quantitatives.describe(include="all")

PM2.5 (Particules fines) : La moyenne observée est de 10.15 µg/m³, bien en dessous du seuil réglementaire de 25 µg/m³. Cependant, l'écart-type de 7.93 indique une forte variabilité. Le maximum atteint 190.33 µg/m³, suggérant des pics de pollution importants. La médiane (8.00) étant inférieure à la moyenne, la distribution est asymétrique vers la droite.

Vent (Vitesse) : Vitesse moyenne de 3.04 m/s, relativement modérée. L'écart-type de 1.82 m/s indique une variabilité normale. Les vitesses varient de 0 m/s (calme plat) à 17.8 m/s (venteux).

Température : La température moyenne observée est de 13°C. L'écart-type de 7.2°C suggère des variations saisonnières modérées. Les valeurs négatives (-10.7°C) et positives (40.4°C) indiquent la présence de périodes hivernales et estivales.

Humidité : Humidité moyenne de 75.18%, plutôt élevée. La distribution est concentration entre 63% (Q1) et 90% (Q3), suggérant un climat humide.

Pluie : Précipitations moyennes très faibles (0.08 mm/heure), avec 75% des observations sans pluie. Cela reflète le fait que les précipitations sont des événements localisés dans le temps.

Industries (densité dans rayon 5km) : Moyenne de 5.15 installations par station, avec forte variabilité (6.38). Les zones rurales peuvent avoir 0 industrie, tandis que certains sites près de zones industrielles en comptent jusqu'à 22. C'est une proxy de la pollution locale d'origine structurelle.

On vérifiera dans la suite la pertinence de ces variables dans notre analyse.


### Diagramme en barre des dépassements de seuil de concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_frequence_depassement

print(graphique_frequence_depassement(df, "depasse_seuil_24h"))

Ce graphique illustre la distribution binaire de la conformité réglementaire des concentrations en PM2.5 en Île-de-France sur la période 2021-2025, basée sur le seuil européen de 25 µg/m³. L'analyse révèle une asymétrie marquée : 81.3% des observations restent en dessous du seuil, tandis que 18.7%  le dépassent. Cette distribution montre que, bien que la qualité de l'air générale soit acceptable, des épisodes de pollution significatifs surviennent régulièrement. Le taux de dépassement de 18.7% équivaut à environ 1 jour sur 5.4 où les concentrations excèdent la limite réglementaire.


### Diagramme en barre des stations avec les mesures de concentrations moyennes les plus hautes et les plus basses


In [ ]:
from src.data_preprocessing import graphique_top_bottom_stations

print(graphique_top_bottom_stations(df, n=5, colonne="pm25_brute", station="nom_station"))

Ce graphique en barres horizontales compare les concentrations moyennes de PM2.5 entre les 5 stations les plus polluées (en rouge) et les 5 moins polluées (en vert) en Île-de-France. Les résultats révèlent ce qui suit : Auto A1 Saint-Denis enregistre 14.16 µg/m³ (2.16× supérieur à Zone Rurale SE avec 6.56 µg/m³), un écart de 7.60 µg/m³. Les stations routières dominent le classement des plus polluées (Auto A1, RN20, Bld Périphérique à 13.04-14.16 µg/m³), reflétant l'impact direct du trafic automobile intensif et des émissions d'échappement. À l'inverse, les zones rurales (Zones Rurale SE/Sud/Nord à 6.56-8.05 µg/m³) bénéficient d'une meilleure ventilation atmosphérique et d'une distance aux sources d'émission. Les stations urbaines parisiennes (Paris 1er, Boulevard Haussmann à 9.58-11.23 µg/m³) occupent une position intermédiaire.

### Distribution de concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_histogramme_pm25

print(graphique_histogramme_pm25(df, colonne="pm25_brute"))

Ce graphique superpose l'histogramme de fréquence (barres bleues) et la courbe de densité KDE (ligne rouge) des concentrations en PM2.5, révélant une distribution fortement asymétrique vers la droite. Le pic principal concentre les observations autour de 8 µg/m³ (la médiane), confirmant que la majorité des mesures reste bien en dessous du seuil alerte (ligne pointillée jaune à 25 µg/m³). Cependant, la queue de distribution s'étend jusqu'à environ 200 µg/m³, indiquant des valeurs extrêmes rares mais significatives. Cette forme est caractéristique des données de pollution environnementale où les conditions normales dominent mais des événements exceptionnels (inversions thermiques, épisodes transfrontaliers) génèrent des pics isolés.

### Boxplot de la concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_boxplot_global

print(graphique_boxplot_global(df, colonne="pm25_brute"))

Ce boxplot synthétise la distribution complète des PM2.5 à travers une représentation robuste des quartiles et des valeurs extrêmes. Le seuil alerte (25 µg/m³, ligne pointillée rouge) dépasse largement la boîte, révélant que les dépassements nécessitent une accumulation significative de polluants au-delà des conditions normales. Les points noirs au-dessus de la boîte représentent des valeurs extrêmes. Ces observations confirment les analyses faites jusqu'à présent.

## 2. ANALYSE TEMPORELLE DE LA VARIABLE CIBLE (PM2.5)

### Analyse de l'évolution mensuelle de la concentration de PM2.5

In [ ]:
from src.data_preprocessing import tableau_moyennes_mensuelles, graphique_moyennes_mensuelles

print(tableau_moyennes_mensuelles(df, "pm25_brute", "mois"))
print(graphique_moyennes_mensuelles(df, "pm25_brute", "mois"))

Ce tableau et ce graphique révèlent un cycle saisonnier extrêmement marqué. Les moyennes varient de 13.98 µg/m³ (mars, pic hivernal) à 7.28 µg/m³ (août, creux estival), soit une amplitude saisonnière de 92% (ratio 1.92 entre max et min). L'hiver (janvier-mars) concentre les concentrations les plus élevées (13.51-13.98 µg/m³) avec des écarts-types importants (10.71-11.19), indiquant une grande variabilité intra-mensuelle due aux inversions thermiques intermittentes et au chauffage résidentiel intensif. L'été (juillet-septembre) enregistre les niveaux minimaux (7.28-7.79 µg/m³) avec écarts-types réduits (4.27-5.22), confirmant des conditions stables et favorables à la dispersion. On remarque également des transitions saisonnières nettes : avril marque une chute abrupte de 28%, tandis que novembre et décembre amorçent progressivement la remontée hivernale.

### Analyse de l'évolution saisonnier de la concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_moyennes_saisonnieres

print(graphique_moyennes_saisonnieres(df, "pm25_brute", "saison"))

Ces résultats viennent confirmés les observations faites plus haut.

### Analyse de l'évolution journalière de la concentration de PM2.5

In [ ]:
from src.data_preprocessing import graphique_serie_temporelle

print(graphique_serie_temporelle(df, "pm25_brute", "datetime_debut"))

Cette série temporelle de mesures horaires révèle la dynamique complète de la pollution aux PM2.5 en Île-de-France avec une granularité fine permettant d'identifier cycles saisonniers, événements extrêmes et tendances à long terme. Le graphique exhibe un pattern saisonnier : les concentrations sont systématiquement élevées en hiver (janvier-mars de chaque année) et minimales en été (juillet-septembre). Cette répétition annuelle confirme que la saisonnalité est le principal moteur de la variabilité des PM2.5. Le seuil alerte (25 µg/m³, ligne pointillée rouge) est dépassé, mais ces dépassements ne sont pas uniformément distribués : ils se concentrent massivement en hiver (clusters denses novembre-mars) et disparaissent quasi-complètement en été (juin-septembre).

### Analyse de l'évolution journalière de la concentration de PM2.5 dans la semaine

In [ ]:
from src.data_preprocessing import tableau_moyennes_jour_semaine

print(tableau_moyennes_jour_semaine(df, "pm25_brute", "jour_semaine"))

In [ ]:
from src.data_preprocessing import graphique_boxplot_jour_semaine

print(graphique_boxplot_jour_semaine(df, "pm25_brute", "jour_semaine"))

In [ ]:
from src.data_preprocessing import graphique_cycle_diurne

print(graphique_cycle_diurne(df, "pm25_brute", "heure"))

On remarque que le profil horaire révèle un cycle diurne très faible. Les concentrations fluctuent faiblement entre 8.8 µg/m³ (14h-16h) et 11.3 µg/m³ (19h-20h). Les deux zones colorées (rush matin 6h-8h et rush soir 17h-20h) montrent que le trafic automobile a un faible impact sur les PM2.5.Les boxplots et le tableau confirment une variabilité hebdomadaire très modérée. Le mercredi est légèrement plus pollué (moyenne 10.70, écart-type 8.80) tandis que le dimanche est le moins pollué (9.45, 7.27), mais cet écart de 1.25 µg/m³ (11%) est minime comparé à la saisonnalité (92%). On ne remarque pas de signature week-end marquée : le samedi (10.12) et dimanche (9.45) ne sont pas beaucoup plus propres que le mercredi (10.70). Les outliers sont distribués uniformément sur tous les jours, indiquant que les événements extrêmes ne sont pas liés au cycle hebdomadaire. Cette quasi-absence de signal jour-de-semaine suggère que le signal prédictif principal reste saisonnier.



### Analyse de l'effet jour-heure sur la concentration des PM2.5

In [ ]:
from src.data_preprocessing import graphique_heatmap_heure_jour

print(graphique_heatmap_heure_jour(df, "pm25_brute", "heure", "jour_semaine"))

Cette heatmap révèle des foyers de pollution significatifs identifiables par les teintes orange-rouge (11.0-12.0 µg/m³), formant un pattern cohérent et répétitif sur 5 jours consécutifs (mardi à samedi). Le mardi affiche les zones rouges les plus intenses, avec une première hausse importante de 6h à 13h (rouge-orange soutenu, 11.5-12.0 µg/m³), particulièrement accentuée entre 9h-12h (pics rouges clairs), suivie d'une deuxième surge soirée entre 18h-22h (orange-rouge, 11.0-11.5 µg/m³). Le mercredi présente le pattern le plus extrême et prolongé : une bande rouge quasi-continue et soutenue de 5h à 14h. Le jeudi maintient des teintes orange-rouge importants entre 5h et 13h, bien que légèrement moins intenses que mercredi. Le vendredi conserve une bande orange nettement visible de 6h à 13h (11.0-11.5 µg/m³) et une nouvelle zone orange-rouge entre 18h-22h (11.0-11.5 µg/m³), prolongeant le pattern de 4 jours. Le samedi affiche encore des teintes orange-rouge substantiels de 17h à 22h (11.0-11.5 µg/m³), marquant la dernière manifestation du phénomène avant la chute drastique vers dimanche. Ces teintes chaudes systématiques de mardi à samedi constituent un signal prédictif majeur et justifient pleinement l'inclusion de variables jour-de-semaine et heure dans le modèle, contrairement à ce que suggérait la faiblesse apparente des signaux isolés.

## 3. ANALYSE SPATIALE DE LA VARIABLE CIBLE (PM2.5)

In [ ]:
from src.data_preprocessing import tableau_comparaison_stations

print(tableau_comparaison_stations(df, "nom_station", "pm25_brute", "depasse_seuil_24h", "datetime_debut", "nb_installations_5km"))

In [ ]:
from src.data_preprocessing import graphique_boxplot_stations

print(graphique_boxplot_stations(df, "nom_station", "pm25_brute"))

In [ ]:
from src.data_preprocessing import graphique_bar_depassements_stations

print(graphique_bar_depassements_stations(df, "nom_station", "depasse_seuil_24h"))

Ces sorties révèlent des disparités spatiales extrêmes et systématiques. Les concentrations moyennes varient fortement : Auto A1 Saint-Denis atteint 14.16 µg/m³ (plus polluée) tandis que Zone Rurale SE affiche 6.56 µg/m³ (moins polluée). Les stations routières dominent le classement supérieur : Auto A1 (14.16), RN20-Montlhéry (13.74), Bld Périphérique Est (13.04), toutes concentrant le trafic intensif automobile et affichant les écarts-types les plus élevés (8.41-8.53), traduisant une grande variabilité intra-station due aux fluctuations du trafic. Les stations urbaines parisiennes (PARIS 1er Les Halles 11.23, Av Champs-Élysées 10.34, Boulevard Haussmann 9.58) occupent une position intermédiaire, bénéficiant de mesures de régulation du trafic (zones piétonnes, limitation vitesse) tout en restant exposées aux émissions. Les zones industrielles (Gennevilliers 10.26, Bobigny 9.63, Gonesse 9.81) affichent des concentrations modérées, suggérant que la pollution industrielle pèse moins que la proximité routière. Les zones rurales (Zones Rurale Nord/Sud/SE à 6.56-8.05) constituent un groupe homogène et nettement moins pollué, avec écarts-types minimaux (5.51-6.76), témoignant de conditions atmosphériques stables et loin des sources d'émission.

## 4. ANALYSE DES CORRÉLATIONS (VARIABLES MÉTÉO ET D'INFRASTRUCTURES) AVEC LA VARIABLE CIBLE PM2.5

In [ ]:
from src.data_preprocessing import graphique_heatmap_correlation
list_variable = ["pm25_brute", "vent_vitesse_ms", "vent_direction_deg", "temperature_c", "humidite_pct", "pluie_mm"]
print(graphique_heatmap_correlation(df, list_variable))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "vent_vitesse_ms", "pm25_brute", "Vitesse du vent", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "vent_vitesse_ms"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "vent_direction_deg", "pm25_brute", "Direction du vent", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "vent_direction_deg"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "temperature_c", "pm25_brute", "Température", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "temperature_c"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "humidite_pct", "pm25_brute", "Humidité", "PM2.5 (µg/m³)"))

In [ ]:
from src.data_preprocessing import histogramme_densité_superposée

print(histogramme_densité_superposée(df, "humidite_pct"))

In [ ]:
from src.data_preprocessing import graphique_scatter_pm25

print(graphique_scatter_pm25(df, "pluie_mm", "pm25_brute", "Précipitations mm", "PM2.5 (µg/m³)"))

L'analyse de ces graphiques révèle que les corrélations météorologiques avec PM2.5 restent globalement faibles (|r| < 0.3 dans la heatmap), mais avec des disparités importantes selon les variables. La vitesse du vent (r = -0.186) affiche une relation non-linéaire marquée où les concentrations diminuent exponentiellement dès 5 m/s, et les dépassements surviennent massivement dans des conditions calmes (< 2 m/s). La température (r = -0.258) se démarque comme le prédicteur météorologique le plus fort, avec une séparation nette entre groupes : les dépassements (rose) se concentrent à -10 à 10°C (hiver), tandis que l'absence de dépassement (bleu) s'étend vers 40°C, reflétant fidèlement la saisonnalité hivernale. Les précipitations (r = -0.162) montrent un effet dès 1-2 mm; les concentrations chutent, et au-delà de 15 mm, elles disparaissent quasi-complètement. La direction du vent (r = -0.195) affiche une corrélation surprenante mais faiblement discriminante. L'humidité (r = 0.030) révèle une corrélation quasi-nulle, les deux groupes (avec/sans dépassement) ayant des distributions identiques, bien que la variance s'accroisse à humidités élevées. En synthèse, température et vitesse du vent sont les véritables leviers prédictifs, tandis que direction et humidité ont un pouvoir univarié limité.

### Comparaison spatiale de la concentration de PM2.5 et de la densité des industries

In [ ]:
from src.data_preprocessing import tracer_comparaison_carto

print(tracer_comparaison_carto(df, 2024, "nom_station", "pm25_brute", "nb_installations_5km", "lat", "lon", "annee"))

Bien que les données ne soient pas disponibles pour l'année 2025, on remarque qu'en 2024 les concentrations de PM2.5 correspondent effectivement aux zones avec une densité industrielle plus élévées. Ce constat vient confirmer l'analyse spatiale réalisée.

## 5. ANALYSE DE PERSISTANCE (AUTO-CORRÉLATION)

In [ ]:
from src.data_preprocessing import graphique_acf_pm25

print(graphique_acf_pm25(df, "pm25_brute", "datetime_debut"))

In [ ]:
from src.data_preprocessing import graphique_pacf_pm25

print(graphique_pacf_pm25(df, "pm25_brute", "datetime_debut"))

L’analyse des fonctions d’autocorrélation (ACF) et d’autocorrélation partielle (PACF) des concentrations horaires de PM2.5 met en évidence une forte dépendance temporelle des données. L’ACF montre une décroissance très lente avec des coefficients élevés même à des retards importants, ce qui indique une forte persistance des niveaux de pollution et suggère une série non stationnaire, ce qui est cohérent avec la variabilité importante observée dans l'analyse temporelle. La PACF, quant à elle, présente des pics significatifs aux premiers retards, suivis d’une décroissance progressive, ce qui suggère que la dynamique de la série peut être bien capturée par un modèle autorégressif de faible ordre. Ces résultats confirment que les concentrations de PM2.5 dépendent fortement des valeurs passées à court terme, traduisant des phénomènes d’accumulation et de dispersion progressive des particules dans l’atmosphère.

# Conclusion

En conclusion, l’analyse des données de PM2.5 en Ile-de-France met en évidence des dynamiques spatio-temporelles marquées, avec une forte saisonnalité, de faibles variations hebdomadaires et des disparités géographiques significatives. Ces résultats confirment que la pollution aux particules fines est influencée à la fois par des facteurs météorologiques (température, vent, humidité), des facteurs temporels (saisons) et des caractéristiques spatiales (densité industrielle, proximité des axes routiers). Ainsi, le choix des variables explicatives retenues pour la modélisation apparaît particulièrement pertinent, car il permet de capturer à la fois les effets de persistance observés dans les séries temporelles, les déterminants environnementaux et les spécificités locales.